# Scaled Dot-Product Attention

MHA 的核心计算单元，单个 head 内部的 attention 计算。

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q (Query)** — 当前位置「想找什么」
- **K (Key)** — 每个位置「我是什么」
- **V (Value)** — 每个位置「我携带的信息」
- **$d_k$** — K 的维度，用于缩放防止点积过大

计算步骤：
1. $QK^T$ — Q 和所有 K 做点积，得到相似度分数矩阵
2. $/ \sqrt{d_k}$ — 缩放，防止 softmax 进入梯度消失区
3. softmax — 归一化为概率分布（权重之和为 1）
4. $\times V$ — 按权重对 V 加权求和，得到输出

In [13]:
import torch
import math

In [14]:
def scaled_dot_product_attention(Q, K, V):
    d_k = K.size(-1)                                      # K 的最后一维维度
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)  # (B, T_q, T_k)
    weights = torch.softmax(scores, dim=-1)               # 对 key 方向归一化
    return torch.bmm(weights, V)                          # (B, T_q, d_v)

## 为什么除以 $\sqrt{d_k}$？

**直觉：点积会随维度增大而变大。**

假设 Q 和 K 的每个分量都是均值为 0、方差为 1 的随机变量，那么它们的点积：

$$q \cdot k = \sum_{i=1}^{d_k} q_i k_i$$

每一项 $q_i k_i$ 的期望为 0，方差为 1，加和后总方差为 $d_k$，标准差为 $\sqrt{d_k}$。

也就是说，**$d_k$ 越大，点积的数值越大**。

---

**问题：点积过大会让 softmax 饱和。**

softmax 的公式是 $\frac{e^{x_i}}{\sum_j e^{x_j}}$，当输入值很大时，最大值对应的概率趋近于 1，其他位置趋近于 0：

```
scores = [0.1, 0.2, 0.3]   → softmax → [0.30, 0.34, 0.36]  # 分布均匀，梯度正常
scores = [10,  20,  30 ]   → softmax → [0.00, 0.00, 1.00]  # 极度集中，梯度消失
```

softmax 饱和后梯度几乎为 0，模型无法正常训练。

---

**解决：除以 $\sqrt{d_k}$ 把方差归一化回 1。**

$$\frac{q \cdot k}{\sqrt{d_k}}$$

---

**为什么是 $\sqrt{d_k}$ 而不是 $d_k$？从方差推导：**

单项 $q_i k_i$ 的方差（$q_i$、$k_i$ 独立，均值为 0，方差为 1）：

$$\text{Var}(q_i k_i) = E[q_i^2] \cdot E[k_i^2] - (E[q_i] \cdot E[k_i])^2 = 1 \cdot 1 - 0 = 1$$

各项独立，方差直接相加，点积总方差：

$$\text{Var}\left(\sum_{i=1}^{d_k} q_i k_i\right) = d_k$$

我们想让归一化后的方差变回 1，关键性质：**常数可以提出来变成平方**

$$\text{Var}(aX) = a^2 \cdot \text{Var}(X)$$

推导：方差定义为 $E[(X-\mu)^2]$，代入 $aX$：

$$\text{Var}(aX) = E[(aX - a\mu)^2] = E[a^2(X-\mu)^2] = a^2 \cdot \text{Var}(X)$$

分别试试除 $\sqrt{d_k}$ 和 $d_k$（即 $a = \frac{1}{\sqrt{d_k}}$ 和 $a = \frac{1}{d_k}$）：

$$\text{Var}\left(\frac{x}{\sqrt{d_k}}\right) = \left(\frac{1}{\sqrt{d_k}}\right)^2 \cdot d_k = \frac{1}{d_k} \cdot d_k = 1 \quad \checkmark$$

$$\text{Var}\left(\frac{x}{d_k}\right) = \left(\frac{1}{d_k}\right)^2 \cdot d_k = \frac{1}{d_k} \quad \text{矫枉过正，}d_k \text{ 越大反而越小}$$

**除以标准差 $\sqrt{d_k}$ 刚好把方差归一化到 1，是最自然的缩放。**

## Self-Attention

Q、K、V 来自同一序列，每个位置可以关注序列中所有其他位置。

## torch.bmm vs 点积

`torch.bmm` 是 **batched matrix multiplication**，不是点积。

```
(B, T_q, d_k) @ (B, d_k, T_k) -> (B, T_q, T_k)
```

结果矩阵中每个元素 `[b, i, j]` 是第 i 个 query 和第 j 个 key 的点积 —— 也就是说，矩阵乘法一次性算出了所有 Q、K 位置两两之间的点积。

> GPU 对矩阵乘法极度优化（CUDA cuBLAS），所以在 PyTorch 中，所有批量点积计算都转换成矩阵乘法来做，而不是逐个向量单独计算。

In [15]:
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)  # (batch=2, seq_len=4, d_k=8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)  # (2, 4, 8)

Output shape: torch.Size([2, 4, 8])


## Cross-Attention

Q 来自一个序列，K/V 来自另一个序列，Q 和 K 的 seq_len 可以不同，但 d_k 必须相同。

两次矩阵乘法收缩的维度不同：

```
scores = Q @ K.T     # (B, T_q, d_k) @ (B, d_k, T_k) -> (B, T_q, T_k)  通过 d_k 收缩
out    = scores @ V  # (B, T_q, T_k) @ (B, T_k, d_v) -> (B, T_q, d_v)  通过 T_k 收缩
```

- 第一次消掉 `d_k`，得到每对 Q-K 的相似度
- 第二次消掉 `T_k`，用相似度权重对 V 加权求和

输出 shape 由 **Q 的 seq_len** 和 **V 的 d_v** 决定：$(B,\\ T_q,\\ d_v)$

现在流行的 LLM（GPT、LLaMA 等）是 **Decoder-only** 架构，没有 Encoder，也就没有 Cross-Attention，只有 Causal Self-Attention。

Cross-Attention 主要活跃在：
- 机器翻译（原始 Transformer）
- 多模态模型（图像 token 作为 K/V，文本作为 Q）— 比如 Flamingo、LLaVA

In [16]:
Q2 = torch.randn(1, 3, 16)  # (batch=1, seq_q=3, d_k=16)
K2 = torch.randn(1, 5, 16)  # (batch=1, seq_k=5, d_k=16)
V2 = torch.randn(1, 5, 32)  # (batch=1, seq_k=5, d_v=32)

out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attention shape:", out2.shape)  # (1, 3, 32)

Cross-attention shape: torch.Size([1, 3, 32])
